[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Migrations


## What you will be able to do

Change a model that has rows behind it, and have the database follow. Set up Alembic against
`SQLModel.metadata`, autogenerate a revision from the models, read what it wrote, and run it. Know
the two things that are SQLModel's rather than Alembic's: the type the generated script names and
does not import, and the annotation that cannot find a class defined further down the file. Build a
database from the migrations alone, and recognize the failures: the missing import, a database that
was built by `create_all` instead, a forward reference that cannot be resolved, and a migration with
nothing in it.


## The idea

### The problem

`SQLModel.metadata.create_all(engine)` creates the tables that are missing. That is all it does, and
it is exactly right until the day a model changes. Add a field to a class, run `create_all` again,
and it looks at the database, finds a table of that name, and leaves it alone. The class now has a
column the table does not, and the first query that mentions it fails.

There is no version of `create_all` that fixes this. Adding a column to a table that holds rows is a
decision, not a deduction: what goes in the column for the rows already there, whether the new rule
is enforced on old rows, what the change should do if it is run twice, and how to undo it. Those
answers have to be written down, in order, and kept beside the code.

That is a migration: a small script with an `upgrade` and a `downgrade`, given a number, applied in
order, and recorded in the database so that a program can tell which of them have run. Alembic
writes and runs them, and it can write most of one by comparing the models with the database.

### What a migration is, and what Alembic does

> A **migration** is a script with **`upgrade()`** and **`downgrade()`**, identified by a revision
> id and pointing at the one before it. **`alembic upgrade head`** runs every script the database
> has not run yet and records each in an `alembic_version` table.
> **`alembic revision --autogenerate`** compares **`target_metadata`**, which for SQLModel is
> **`SQLModel.metadata`**, with the database and writes the difference into a new script, which is
> a draft to read rather than an answer to trust.

### Why it works that way

- **The models are the destination, not the history.** They say what the schema should be now; the
  revisions say how every database gets there from wherever it is.
- **Autogenerate compares, it does not guess.** It sees a column in the metadata and not in the
  database, and writes `add_column`. A column renamed looks to it like one dropped and one added,
  which is why the draft is read before it is run.
- **SQLModel's string type has a name of its own.** A `str` field becomes
  `sqlmodel.sql.sqltypes.AutoString`, and the generated script names that type without importing it,
  which is the one failure in this notebook that belongs to SQLModel rather than to Alembic.
- **The revision runs in a process of its own.** `alembic` imports `env.py`, which imports the
  models, so the classes have to be in a module on disk rather than in a notebook cell.
- **SQLite cannot alter much.** It has added and dropped columns since 3.35, and changing a
  constraint means rebuilding the table, which Alembic does with batch mode.

### Where this shows up

Every project past its first week. The **SQLAlchemy, Deep Dive** guide's Migrations with Alembic
notebook is where Alembic itself is taught: `init`, `env.py`, the autogenerate cycle, batch mode and
`render_as_batch=True` for SQLite's limits. This notebook assumes all of that and spends itself on
what is different when the models are SQLModel's.

### What this notebook covers

- What `create_all` does with a model that changed
- The project on disk, and Alembic pointed at `SQLModel.metadata`
- The first revision, autogenerated and read
- The type the script names and does not import
- A second revision, from an edited `models.py`
- SQLite's limits, in one line
- A database built from the migrations alone, finished
- Four failures, from a missing import to a migration with nothing in it

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import Column, String, create_engine, inspect
from sqlmodel import Field, SQLModel


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
print("the table has      :", [column["name"] for column in inspect(engine).get_columns("hero")])

Hero.__table__.append_column(Column("nickname", String))    # the model gains a column
SQLModel.metadata.create_all(engine)                        # and create_all is asked again
print("the model has      :", [column.name for column in Hero.__table__.columns])
print("the table still has:", [column["name"] for column in inspect(engine).get_columns("hero")])
```

```
the table has      : ['id', 'name']
the model has      : ['id', 'name', 'nickname']
the table still has: ['id', 'name']
```

The model gained a column, `create_all` was asked again, and the table is what it was. That is the
whole of its contract: it creates what is missing and changes nothing that exists, which is why a
change to a model needs something else entirely. Everything below is about what that something is.


## Setup

Ten imports, two packages installed first where they are missing, five helpers, and the models
written to a file.

- `sqlmodel` is the library, and the cell prints its version beside Alembic's. Colab has neither, so
  the cell installs 0.0.42 and 1.20.0 with `pip` where they are missing, and `version` and
  `PackageNotFoundError`, from `importlib.metadata`, find out whether it has to
- `create_engine`, `inspect` and `text`, from `sqlalchemy`, read what the database actually holds,
  which is how every claim in this notebook is checked
- `subprocess`, `sys`, `os` and `shlex` run `alembic` as a command and print the line that was run,
  with the scratch folder's path taken out so that the output is the same on every machine
- `re` cuts a traceback down to the line that names the error, `Path` names the project's files, and
  `shutil` removes the scratch folder at the start and at the end

No model is defined in this notebook. Alembic imports `env.py` in a process of its own, `env.py`
imports the models, and a class defined in a cell is not visible there, so the models live in
`scratch/models.py` and every command that needs them runs in its own Python. That is also why the
notebook can change a model at all: editing a file and running a command again is possible, where
defining the same class twice in one session is not, as the **Relationships** notebook showed.

`alembic` prints the file it wrote and the revisions it ran; `run_python` writes a short program and
runs it; `edit` changes one piece of a file and fails rather than silently do nothing; `revision`
prints a script from its `def upgrade` on, since the header carries the time it was written; and
`columns_of` reads a table's columns from the database.


In [1]:
import os
import re
import shlex
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

for package, pin in (("sqlmodel", "sqlmodel==0.0.42"), ("alembic", "alembic==1.20.0")):
    try:
        version(package)
    except PackageNotFoundError:                                    # Colab has neither: install the pinned versions
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", pin],
                       check=True)

import sqlmodel
from sqlalchemy import create_engine, inspect, text

SCRATCH = Path("scratch")


def alembic(*arguments):
    """Run one alembic command in the scratch folder and print what it said."""
    done = subprocess.run([sys.executable, "-m", "alembic", *arguments], cwd=SCRATCH,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          env={**os.environ, "NO_COLOR": "1", "PYTHONUNBUFFERED": "1",
                               "PYTHONDONTWRITEBYTECODE": "1"})
    lines = done.stdout.replace(f"{SCRATCH.resolve()}{os.sep}", "").splitlines()
    if "Traceback (most recent call last):" in lines:               # the error's own line, not the whole traceback
        named = [line for line in lines if re.match(r"[\w.]+(Error|Exception): ", line)]
        lines = lines[:lines.index("Traceback (most recent call last):")] + ["Traceback ...", named[-1]]
    print("$", shlex.join(["alembic", *arguments]))
    for line in lines:
        if not any(noise in line for noise in ("Context impl", "Will assume", "Please edit",
                                               "setting up autogenerate plugin")):
            print("   ", line)


def run_python(source, name="a_script.py"):
    """Write a small program into the scratch folder, run it in a Python of its own, and print what it said."""
    (SCRATCH / name).write_text(source)
    done = subprocess.run([sys.executable, name], cwd=SCRATCH, capture_output=True, text=True)
    printed = done.stdout.strip() or done.stderr.strip().splitlines()[-1]
    print(printed.replace(f"{SCRATCH.resolve()}{os.sep}", ""))


def edit(path, old, new):
    """Change one piece of a file, and fail rather than silently do nothing."""
    text = path.read_text()
    assert text.count(old) == 1, f"{path.name}: found {text.count(old)} of {old!r}"
    path.write_text(text.replace(old, new))


def revision(name):
    """A revision script from its def upgrade on: the header holds the time it was written."""
    body = (SCRATCH / "migrations" / "versions" / name).read_text()
    print(body[body.index("def upgrade"):body.index("def downgrade")].rstrip())


def columns_of(table, database="heroes.db"):
    """The column names a database has for a table, read from the database itself."""
    engine = create_engine(f"sqlite:///scratch/{database}")
    try:
        return [column["name"] for column in inspect(engine).get_columns(table)]
    finally:
        engine.dispose()

MODELS = """from sqlmodel import Field, Relationship, SQLModel


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")
"""

shutil.rmtree(SCRATCH, ignore_errors=True)                          # a rerun starts from no project at all
SCRATCH.mkdir()
(SCRATCH / "models.py").write_text(MODELS)

print("sqlmodel", sqlmodel.__version__, "| alembic", version("alembic"),
      "| the project:", sorted(path.name for path in SCRATCH.iterdir()))


sqlmodel 0.0.42 | alembic 1.20.0 | the project: ['models.py']


## Worked examples

### What create_all does with a model that changed

The models are on disk, so the whole of this happens in one program, which builds a database from
them, changes a model, and asks again:


In [2]:
run_python('''
from sqlmodel import Session, SQLModel, create_engine, select

from models import Hero, Team

engine = create_engine("sqlite:///heroes.db")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Team(name="Preventers", headquarters="Sharp Tower"))
    session.add(Hero(name="Rusty-Man", secret_name="Tommy Sharp", age=48, team_id=1))
    session.commit()
    print("built and loaded:", len(session.exec(select(Hero)).all()), "hero")
''', name="build.py")

print("columns:", columns_of("hero"))


built and loaded: 1 hero
columns: ['id', 'name', 'secret_name', 'age', 'team_id']


A database with a hero in it. Now the models gain a field, which is the ordinary business of a
project, and `create_all` is asked again:


In [3]:
edit(SCRATCH / "models.py",
     '    team_id: int | None = Field(default=None, foreign_key="team.id")',
     '    nickname: str | None = Field(default=None, max_length=40)\n'
     '    team_id: int | None = Field(default=None, foreign_key="team.id")')

run_python('''
from sqlmodel import Session, SQLModel, create_engine, select

from models import Hero

engine = create_engine("sqlite:///heroes.db")
SQLModel.metadata.create_all(engine)                    # the table is there, so nothing happens
print("columns after create_all:", [column.name for column in Hero.__table__.columns])
with Session(engine) as session:
    print(session.exec(select(Hero)).all())
''', name="again.py")

print("the database still has:", columns_of("hero"))


columns after create_all: ['id', 'name', 'secret_name', 'age', 'nickname', 'team_id']
the database still has: ['id', 'name', 'secret_name', 'age', 'team_id']


The class has six columns and the table has five, `create_all` said nothing about the difference,
and the first query that mentions the model failed with `no such column: hero.nickname`. That is the
whole motivation for what follows, and it is worth noticing how quiet the failure is until a query
runs: nothing at import, nothing at `create_all`, and then a page that will not load.

### The project on disk, and Alembic pointed at SQLModel.metadata

`alembic init` writes the scaffolding, and two edits point it at this project: the database in
`alembic.ini`, and the models in `env.py`. The **SQLAlchemy, Deep Dive** guide's Migrations with
Alembic notebook is where all of this is taken apart; here it is three commands and two edits:


In [4]:
alembic("init", "migrations")

ini = SCRATCH / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///heroes.db",
                      ini.read_text(), flags=re.M))
edit(SCRATCH / "migrations" / "env.py", "target_metadata = None",
     'import sys\n\nsys.path.insert(0, ".")                            # models.py is beside alembic.ini\n'
     "from models import SQLModel\n\ntarget_metadata = SQLModel.metadata")

print("the project:", sorted(path.name for path in SCRATCH.iterdir()))


$ alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
the project: ['__pycache__', 'again.py', 'alembic.ini', 'build.py', 'heroes.db', 'migrations', 'models.py']


`target_metadata = SQLModel.metadata` is the SQLModel-shaped line. Importing `models` is what fills
that metadata in: the tables exist because the classes were defined, so a model nobody imports is a
table Alembic will not know about, and it will cheerfully write a revision that drops it.

### The first revision, autogenerated and read

The database this project has was built by `create_all`, which is not where a migration story
starts. `alembic stamp` is one answer to that, and the second of the Common errors is what happens
without it; here the database is removed, the models go back to what they were before the `nickname`
was added, and the whole schema comes from the migration instead. The nickname then comes back the
way every change should, as a revision of its own.


In [5]:
(SCRATCH / "heroes.db").unlink()
edit(SCRATCH / "models.py",
     "    nickname: str | None = Field(default=None, max_length=40)\n", "")

alembic("revision", "--autogenerate", "-m", "the heroes", "--rev-id", "0001")


$ alembic revision --autogenerate -m 'the heroes' --rev-id 0001
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'team'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_team_name' on '('name',)'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'hero'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_age' on '('age',)'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_name' on '('name',)'
    Generating migrations/versions/0001_the_heroes.py ...  done


In [6]:
revision("0001_the_heroes.py")


def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.create_table('team',
    sa.Column('id', sa.Integer(), nullable=False),
    sa.Column('name', sqlmodel.sql.sqltypes.AutoString(length=50), nullable=False),
    sa.Column('headquarters', sqlmodel.sql.sqltypes.AutoString(length=60), nullable=False),
    sa.PrimaryKeyConstraint('id')
    )
    op.create_index(op.f('ix_team_name'), 'team', ['name'], unique=False)
    op.create_table('hero',
    sa.Column('id', sa.Integer(), nullable=False),
    sa.Column('name', sqlmodel.sql.sqltypes.AutoString(length=50), nullable=False),
    sa.Column('secret_name', sqlmodel.sql.sqltypes.AutoString(length=60), nullable=False),
    sa.Column('age', sa.Integer(), nullable=True),
    sa.Column('team_id', sa.Integer(), nullable=True),
    sa.ForeignKeyConstraint(['team_id'], ['team.id'], ),
    sa.PrimaryKeyConstraint('id')
    )
    op.create_index(op.f('ix_hero_age'), 'hero', ['age'], u

Two tables, three indexes and a foreign key, none of it written by hand. The line to look at is the
type of every string column:

`sa.Column('name', sqlmodel.sql.sqltypes.AutoString(length=50), nullable=False)`

`AutoString` is SQLModel's, not SQLAlchemy's: it is the type a `str` field becomes, and it renders
as `VARCHAR` on every database. Alembic wrote its full name into the script, correctly, and wrote no
import for it, which is the next section.

### The type the script names and does not import

Running the revision as written is the fastest way to see the problem:


In [7]:
alembic("upgrade", "head")


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0001, the heroes
    Traceback ...
    NameError: name 'sqlmodel' is not defined


`NameError: name 'sqlmodel' is not defined`, from a script Alembic wrote itself. The generated file
imports `sqlalchemy as sa` and `from alembic import op`, and nothing else; the `sqlmodel` in
`sqlmodel.sql.sqltypes.AutoString` is a name the script never had.

Two lines fix it, one for this script and one for every script after it:


In [8]:
edit(SCRATCH / "migrations" / "versions" / "0001_the_heroes.py",
     "import sqlalchemy as sa", "import sqlalchemy as sa\nimport sqlmodel")
edit(SCRATCH / "migrations" / "script.py.mako",
     "import sqlalchemy as sa", "import sqlalchemy as sa\nimport sqlmodel")

alembic("upgrade", "head")
print("columns:", columns_of("hero"))


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0001, the heroes
columns: ['id', 'name', 'secret_name', 'age', 'team_id']


The revision ran, the tables are there, and `alembic_version` now holds `0001`. The second edit is
the one that matters in a project: `script.py.mako` is the template every generated revision is
written from, so adding the import there means never meeting this again. An import that a particular
revision does not need costs nothing.

### A second revision, from an edited models.py

Now the change that started this notebook, made the way a project makes one: edit the models, and
autogenerate the revision that takes every database from where it is to where they are:


In [9]:
edit(SCRATCH / "models.py",
     '    team_id: int | None = Field(default=None, foreign_key="team.id")',
     '    nickname: str | None = Field(default=None, max_length=40)\n'
     '    team_id: int | None = Field(default=None, foreign_key="team.id")')

alembic("revision", "--autogenerate", "-m", "a nickname", "--rev-id", "0002")
revision("0002_a_nickname.py")


$ alembic revision --autogenerate -m 'a nickname' --rev-id 0002
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'hero.nickname'
    Generating migrations/versions/0002_a_nickname.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.add_column('hero', sa.Column('nickname', sqlmodel.sql.sqltypes.AutoString(length=40), nullable=True))
    # ### end Alembic commands ###


In [10]:
alembic("upgrade", "head")
print("columns:", columns_of("hero"))


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, a nickname
columns: ['id', 'name', 'secret_name', 'age', 'team_id', 'nickname']


One line in the upgrade, one column in the table, and the import was there this time because the
template has it. The change came from editing the models file rather than from redefining a class,
which is the only way it can come in a notebook and the way it always comes in a project.

### SQLite's limits, in one line

SQLite has added and dropped columns since version 3.35, which is what the two revisions above
needed. What it cannot do is change a column or a constraint in place: the table has to be rebuilt,
its rows copied, and the old one dropped. Alembic does that with `batch_alter_table`, switched on by
`render_as_batch=True` in `env.py`, and it needs every constraint to have a name, which is what the
convention in the **sa_column and __table_args__** notebook is for. The
**SQLAlchemy, Deep Dive** guide's Migrations with Alembic notebook does that work in full, on the
same version of Alembic, and nothing about it is different for SQLModel.

### A database built from the migrations alone, finished

The pieces of this notebook in one thing worth checking: a database nobody has ever run `create_all`
against, built from the revisions in order, and holding exactly what the models describe:


In [11]:
ini.write_text(ini.read_text().replace("sqlite:///heroes.db", "sqlite:///fresh.db"))
alembic("upgrade", "head")
alembic("current")

run_python('''
from sqlmodel import Session, SQLModel, create_engine, select

from models import Hero, Team

engine = create_engine("sqlite:///fresh.db")            # built by the migrations, and never by create_all
with Session(engine) as session:
    session.add(Team(name="Z-Force", headquarters="Sister Margaret's Bar"))
    session.add(Hero(name="Deadpond", secret_name="Dive Wilson", nickname="Pond", team_id=1))
    session.commit()
    hero = session.exec(select(Hero)).one()
    print("wrote and read:", hero.name, "of the", hero.team.name, "| nickname:", hero.nickname)
''', name="use_fresh.py")


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0001, the heroes
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, a nickname
$ alembic current
    0002 (head)
wrote and read: Deadpond of the Z-Force | nickname: Pond


The models and the database agree, and nothing built the schema but the two revisions. That is the
test worth running on any project that has been using `create_all`: if a fresh database from
`upgrade head` does not match the models, the migrations have drifted from the code, and it is
better to find that out on a machine than on a server.

### Where each part came from

| In the last cell | What it relies on | The section that showed it |
|---|---|---|
| `alembic upgrade head` on a new file | revisions that describe the whole schema | The first revision, autogenerated and read |
| the `nickname` column being there | a second revision from an edited `models.py` | A second revision |
| the revisions running at all | `import sqlmodel` in the script and in the template | The type the script names |
| `target_metadata = SQLModel.metadata` | Alembic comparing the models with the database | The project on disk |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/12-migrations-solutions.ipynb).

**1.** Print the revisions Alembic has, and which one the fresh database is on, with
`alembic history` and `alembic current`.


In [12]:
# your code here


**2.** Add a `motto` field of at most 80 characters to `Team` in `models.py`, autogenerate revision
`0003`, and print the `upgrade` it wrote.


In [13]:
# your code here


**3.** Run that revision and show the column is in the database.


In [14]:
# your code here


**4.** Undo it with `alembic downgrade -1`, and show the column is gone and the current revision is
`0002`.


In [15]:
# your code here


**5.** Autogenerate a revision when nothing has changed, print what it wrote, and then delete the
file.


In [16]:
# your code here


**6.** Write a program that builds another database from the migrations, writes a team and reads it
back, and run it. `alembic.ini` names the database, so it has to be pointed at the new file first.


In [17]:
# your code here


## Common errors

### NameError: name 'sqlmodel' is not defined


In [18]:
edit(SCRATCH / "migrations" / "script.py.mako",                     # the template, back as Alembic wrote it
     "import sqlalchemy as sa\nimport sqlmodel", "import sqlalchemy as sa")
edit(SCRATCH / "models.py", '    headquarters: str = Field(max_length=60)',
     '    headquarters: str = Field(max_length=60)\n    motto: str | None = Field(default=None, max_length=80)')

alembic("revision", "--autogenerate", "-m", "a motto", "--rev-id", "0003")
alembic("upgrade", "head")


$ alembic revision --autogenerate -m 'a motto' --rev-id 0003
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'team.motto'
    Generating migrations/versions/0003_a_motto.py ...  done
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0002 -> 0003, a motto
    Traceback ...
    NameError: name 'sqlmodel' is not defined


The same failure as the worked example, and this is where it usually arrives: not on the first
revision, which somebody fixed by hand, but on the tenth, written months later by a template nobody
went back to. The message names no file that is yours, and the line it points at is one Alembic
wrote.

The fix is the template, once and for the life of the project:


In [19]:
edit(SCRATCH / "migrations" / "script.py.mako",
     "import sqlalchemy as sa", "import sqlalchemy as sa\nimport sqlmodel")
edit(SCRATCH / "migrations" / "versions" / "0003_a_motto.py",
     "import sqlalchemy as sa", "import sqlalchemy as sa\nimport sqlmodel")

alembic("upgrade", "head")
print("team columns:", columns_of("team", "fresh.db"))              # alembic.ini names fresh.db now


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0002 -> 0003, a motto
team columns: ['id', 'name', 'headquarters', 'motto']


### sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) table team already exists


In [20]:
run_python('''
from sqlmodel import SQLModel, create_engine

from models import Hero, Team

SQLModel.metadata.create_all(create_engine("sqlite:///started.db"))
print("create_all built the tables in started.db")
''', name="start_with_create_all.py")

ini.write_text(ini.read_text().replace("sqlite:///fresh.db", "sqlite:///started.db"))
alembic("upgrade", "head")


create_all built the tables in started.db
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0001, the heroes
    Traceback ...
    sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) table team already exists


This is the shape every project has: the tables were made with `create_all` long before anybody
added Alembic, so the database has the schema and no record of a single revision. `upgrade head`
therefore starts at the beginning and tries to create tables that are already there.

`alembic stamp` is the answer: it writes a revision into `alembic_version` without running anything,
which says that the database is already at that point.


In [21]:
alembic("stamp", "head")
alembic("current")
alembic("upgrade", "head")


$ alembic stamp head
    INFO  [alembic.runtime.migration] Running stamp_revision  -> 0003
$ alembic current
    0003 (head)
$ alembic upgrade head


Stamping claims the database matches the revision named, and nothing checks that claim, so it is
worth being sure of before doing it. The check is the last worked example: build a fresh database
from the migrations, and compare.

### sqlalchemy.exc.InvalidRequestError: When initializing mapper Mapper[Hero(hero)], expression 'Team | None' failed to locate a name ('Team | None')


In [22]:
run_python('''
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: "Team | None" = Relationship(back_populates="heroes")     # the whole annotation in quotes


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    heroes: list[Hero] = Relationship(back_populates="team")


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    session.get(Hero, 1)
''', name="forward.py")


sqlalchemy.exc.InvalidRequestError: When initializing mapper Mapper[Hero(hero)], expression 'Team | None' failed to locate a name ('Team | None'). If this is a class name, consider adding this relationship() to the <class '__main__.Hero'> class after both dependent classes have been defined.


A models file has an order, and a class named before it is defined is a problem Python solves with
quotes. What the quotes may hold is the class name, and not the whole annotation: SQLAlchemy looks
up what is inside them as a name, and there is no class called `Team | None`.

`Optional["Team"]` is the form that works, with the class name alone quoted, and
`if TYPE_CHECKING:` is what gives a type checker the real class without importing it at run time:


In [23]:
run_python('''
from typing import TYPE_CHECKING, Optional

from sqlmodel import Field, Relationship, Session, SQLModel, create_engine

if TYPE_CHECKING:                                       # read by a type checker, never run
    from forward_fixed import Team


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Optional["Team"] = Relationship(back_populates="heroes")


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    heroes: list[Hero] = Relationship(back_populates="team")


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Team(name="Preventers", heroes=[Hero(name="Rusty-Man")]))
    session.commit()
    print("it works:", session.get(Hero, 1).team.name)
''', name="forward_fixed.py")


it works: Preventers


An unquoted name for a class defined further down the same file is a different matter, and what it
does depends on the version of Python: up to 3.13 the annotation is evaluated as the class body runs
and raises `NameError` there, and from 3.14 annotations are evaluated only when something asks for
them, so the file imports and the same mistake surfaces later. Quoting the class name works on every
version, which is why the models files in this guide do it.

### No error, and a revision with nothing in it


In [24]:
alembic("revision", "--autogenerate", "-m", "nothing changed", "--rev-id", "0004")
revision("0004_nothing_changed.py")


$ alembic revision --autogenerate -m 'nothing changed' --rev-id 0004
    Generating migrations/versions/0004_nothing_changed.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    pass
    # ### end Alembic commands ###


Autogenerate found no difference and wrote a revision that does nothing, and it is a real revision:
it has an id, it is in the chain, and `upgrade head` will run it and record it. That is harmless and
untidy, and the reason it happens is usually that the change being looked for is one autogenerate
does not see.

What it does not see is worth knowing: a column renamed looks like one dropped and one added, a
`CHECK` constraint on SQLite is not compared, and anything in a table Alembic was not told about is
invisible. The answer to all of those is the same: read the draft, and write by hand what it missed.


In [25]:
(SCRATCH / "migrations" / "versions" / "0004_nothing_changed.py").unlink()
alembic("history")


$ alembic history
    0002 -> 0003 (head), a motto
    0001 -> 0002, a nickname
    <base> -> 0001, the heroes


Deleting a revision that has not run anywhere is safe, and deleting one that has run on a database
somewhere is not, since that database now names a revision nobody has.

Last, this cell removes the scratch folder with the project, the migrations and the databases in it:


In [26]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())


scratch still there: False


## Recap

- `create_all` creates missing tables and changes nothing that exists, so a model that gained a
  field leaves the database behind and the next query fails.
- Alembic compares `SQLModel.metadata` with the database and writes a revision; the models must be
  in a module `env.py` can import, which is why they live in a file rather than in a cell.
- A generated revision names `sqlmodel.sql.sqltypes.AutoString` for every string column and imports
  nothing: add `import sqlmodel` to the script, and to `script.py.mako` for every script after it.
- A database built by `create_all` has no revision recorded, so `alembic stamp head` is what says it
  is already up to date.
- Quote the class name alone in a forward reference, `Optional["Team"]`, and import it under
  `if TYPE_CHECKING:` for the type checker.


## What is next

The **SQLModel in FastAPI** notebook puts all of it behind routes: a session that belongs to one
request, the create and public models as what a route takes and answers with, the relationship that
fails when the response is built after the session closed, and the two response models that refer to
each other.


---

&#8592; **Previous:** [sa_column and __table_args__](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/11-sa-column-and-table-args.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [SQLModel in FastAPI](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/13-sqlmodel-in-fastapi.ipynb) &#8594;
